### Transform Orders Data - Explode Arrays

 1. Access elements from JSON Object
 2. Deduplicate Array Elements
 3. Explode Arrays
 4. Write the Transformed Data to Silver Schema

### 1. Access elements from JSON Object

In [0]:
dfPyOrders = spark.table("gizmobox_sivan.silver.py_orders_json")
display(dfPyOrders)

In [0]:
dfPyOrders.select(    
    dfPyOrders.json_value.order_id.alias("order_id"),    
    dfPyOrders.json_value.order_status.alias("order_status"),
    dfPyOrders.json_value.order_date.alias("order_date"),
    dfPyOrders.json_value.payment_method.alias("payment_method"),
    dfPyOrders.json_value.total_amount.alias("total_amount"),
    dfPyOrders.json_value.transaction_timestamp.alias("transaction_timestamp"),
    dfPyOrders.json_value.customer_id.alias("customer_id"),  
    dfPyOrders.json_value.items.alias("items")
).display()


### 2. Deduplicate Array Elements

[function - array_distinct](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/array_distinct)

In [0]:
from pyspark.sql import functions as f

dfDistinctItemOrders = (
    dfPyOrders.select(    
        dfPyOrders.json_value.order_id.alias("order_id"),    
        dfPyOrders.json_value.order_status.alias("order_status"),
        dfPyOrders.json_value.order_date.alias("order_date"),
        dfPyOrders.json_value.payment_method.alias("payment_method"),
        dfPyOrders.json_value.total_amount.alias("total_amount"),
        dfPyOrders.json_value.transaction_timestamp.alias("transaction_timestamp"),
        dfPyOrders.json_value.customer_id.alias("customer_id"),  
        f.array_distinct(dfPyOrders.json_value.items).alias("items") #Ex: In prev statement, 10th row was having duplicate items, 2nd row was having 2 distinct items
    )
)

display(dfDistinctItemOrders)


### 3. Explode Arrays

Function [explode](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/explode)

In [0]:
from pyspark.sql import functions as f

dfExplodeOrders = (
    dfDistinctItemOrders.select(    
        dfDistinctItemOrders.order_id.alias("order_id"),    
        dfDistinctItemOrders.order_status.alias("order_status"),
        dfDistinctItemOrders.order_date.alias("order_date"),
        dfDistinctItemOrders.payment_method.alias("payment_method"),
        dfDistinctItemOrders.total_amount.alias("total_amount"),
        dfDistinctItemOrders.transaction_timestamp.alias("transaction_timestamp"),
        dfDistinctItemOrders.customer_id.alias("customer_id"),  
        f.explode(dfDistinctItemOrders.items).alias("item")        
    )
)

dfExplodeOrders.display()

In [0]:
from pyspark.sql import functions as f
dfFinalOrders = (
    dfExplodeOrders.select(    
        dfExplodeOrders.order_id.alias("order_id"),    
        dfExplodeOrders.order_status.alias("order_status"),
        dfExplodeOrders.order_date.alias("order_date"),
        dfExplodeOrders.payment_method.alias("payment_method"),
        dfExplodeOrders.total_amount.alias("total_amount"),
        dfExplodeOrders.transaction_timestamp.alias("transaction_timestamp"),
        dfExplodeOrders.customer_id.alias("customer_id"),          
        f.col('item.category'),    
        f.col('item.details'),
        f.col('item.details.brand'),
        f.col('item.details.color'),
        f.col('item.item_id'),
        f.col('item.name'), # Only for this f.col() is required because of the column name
        f.col('item.price'),
        f.col('item.quantity')      
    )
)

dfFinalOrders.display()


In [0]:
dfFinalOrders.writeTo("gizmobox_sivan.silver.py_orders").createOrReplace()

In [0]:
spark.table("gizmobox_sivan.silver.py_orders").display()